In [ ]:
import os
import kagglehub

os.environ["KAGGLEHUB_CACHE"] = r"D:\KAGGLE_DATASETS"

dataset_path = kagglehub.dataset_download("quandang/vietnamese-foods")

print("Dataset saved to:", dataset_path)

In [ ]:
print(os.listdir(dataset_path))

In [ ]:
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    print(item, "=>", item_path, os.path.isdir(item_path))

In [ ]:
import os

train_dir = r"D:\KAGGLE_DATASETS\datasets\quandang\vietnamese-foods\versions\11\Images\Train"
val_dir = r"D:\KAGGLE_DATASETS\datasets\quandang\vietnamese-foods\versions\11\Images\Validate"
test_dir = r"D:\KAGGLE_DATASETS\datasets\quandang\vietnamese-foods\versions\11\Images\Test"

print(os.listdir(train_dir)[:20])
print("Số class:", len(os.listdir(train_dir)))

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dropout,
    Dense, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 80
MODEL_NAME = "model.keras"

selected_classes = [
    "Banh cuon",
    "Banh mi",
    "Banh xeo",
    "Bun dau mam tom",
    "Banh canh"
]

NUM_CLASSES = len(selected_classes)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    brightness_range=(0.9, 1.1),
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=selected_classes,
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=selected_classes,
    shuffle=False
)

print(train_generator.class_indices)

In [ ]:
num_classes = train_generator.num_classes

model = Sequential([
    Conv2D(16, (3, 3), activation="relu", padding="same",
           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.2),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    GlobalAveragePooling2D(),

    Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    Dropout(0.4),

    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
callbacks = [
    ModelCheckpoint(
        MODEL_NAME,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
import os
from tensorflow.keras.models import load_model

if os.path.exists(MODEL_NAME):
    try:
        model = load_model(MODEL_NAME)
        print(f"Loaded model from {MODEL_NAME}")
    except Exception as e:
        print(f"Failed to load model: {e}")
        print("Training from scratch.")
else:
    print(f"No model found at {MODEL_NAME}; training from scratch.")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.legend()
plt.title("Loss")

plt.show()